# Modifying ONNX Graphs: Deep Dive

This notebook covers ONNX graph modification techniques: formal rewrite rules,
node insertion/deletion invariants, operator replacement, model merging, and safe rewrite patterns.

## 1. Graph Rewrite Rules

A **graph rewrite rule** transforms a subgraph matching a pattern into a replacement:

$$\text{LHS pattern} \to \text{RHS replacement}$$

Given graph $G$ containing subgraph matching pattern $p$, substituting replacement $r$:

$$G[p/r] = G'$$

A rewrite is **valid** iff semantic equivalence holds:

$$\forall x \in \mathcal{X}: f_G(x) = f_{G'}(x)$$

### Preconditions for Safe Rewrites

1. **Pattern matching**: LHS exactly matches a subgraph (structure + attributes)
2. **Type compatibility**: Replacement I/O types match surrounding graph
3. **Shape preservation**: Output shapes consistent for downstream consumers
4. **No side effects**: No observable behavior differences introduced

In [ ]:
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper, checker, shape_inference, compose
import onnxruntime as ort
import copy

print(f"ONNX version: {onnx.__version__}")
print(f"ONNX Runtime version: {ort.__version__}")

## 2. Node Insertion/Deletion Invariants

Every valid ONNX graph must satisfy these structural invariants:

**Inv 1** — Input Coverage: $\forall n \in V: \text{inputs}(n) \subseteq \text{produced}(G) \cup \text{graph\_inputs}(G)$

**Inv 2** — Output Reachability: $\forall o \in \text{graph\_outputs}(G): o \in \text{produced}(G)$

**Inv 3** — Acyclicity: $\nexists \text{ path } n_1 \to n_2 \to \cdots \to n_k \to n_1$

**Inv 4** — Name Uniqueness: $|\text{all\_names}(G)| = |\text{graph\_inputs}(G)| + |\text{produced}(G)|$

```
┌─────────────────────────────────────────┐
│       Invariant Checking Flow           │
├─────────────────────────────────────────┤
│  [Modify Graph]                         │
│       ▼                                 │
│  [Check Inv 1: Input Coverage]          │
│       ▼                                 │
│  [Check Inv 2: Output Reachable]        │
│       ▼                                 │
│  [Check Inv 3: No Cycles]              │
│       ▼                                 │
│  [Check Inv 4: Unique Names]            │
│       ▼                                 │
│  [onnx.checker.check_model()]           │
│       │                                 │
│  ┌────┴────┐                            │
│  ▼         ▼                            │
│ VALID    INVALID → Fix & Retry          │
└─────────────────────────────────────────┘
```

In [ ]:
def check_invariants(model):
    graph = model.graph
    inputs = {i.name for i in graph.input}
    inits = {i.name for i in graph.initializer}
    produced = {o for n in graph.node for o in n.output if o}
    available = inputs | inits | produced

    for node in graph.node:
        for inp in node.input:
            if inp and inp not in available:
                return False, f"Inv1: '{inp}' not produced"
    for out in graph.output:
        if out.name not in produced and out.name not in inputs:
            return False, f"Inv2: '{out.name}' unreachable"
    all_names = [i.name for i in graph.input] + [o for n in graph.node for o in n.output if o]
    if len(all_names) != len(set(all_names)):
        return False, "Inv4: duplicate names"
    return True, "All invariants satisfied"


def validate_model(model, msg=""):
    ok, detail = check_invariants(model)
    prefix = f"[{msg}] " if msg else ""
    if not ok:
        print(f"{prefix}FAILED: {detail}")
        return False
    try:
        checker.check_model(model)
        print(f"{prefix}Valid ✓")
        return True
    except Exception as e:
        print(f"{prefix}Checker error: {e}")
        return False

## 3. Node Insertion Pattern

Given edge $(u, v)$ carrying value $t$, insert node $w$:
1. Create new value $t'$
2. Set $w$: input=$t$, output=$t'$
3. Replace all consumer references to $t$ with $t'$

```
BEFORE:                      AFTER:

 ┌───┐   t   ┌───┐          ┌───┐   t   ┌───┐  t'  ┌───┐
 │ u │──────►│ v │          │ u │──────►│ w │─────►│ v │
 └───┘       └───┘          └───┘       └───┘      └───┘
```

If $t$ has **multiple consumers**, all must be updated to use $t'$.

In [ ]:
def build_simple_model():
    X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
    Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 3])
    W = numpy_helper.from_array(np.random.randn(4, 3).astype(np.float32), name='W')
    B = numpy_helper.from_array(np.random.randn(3).astype(np.float32), name='B')
    nodes = [
        helper.make_node('MatMul', ['X', 'W'], ['mm_out'], name='matmul_0'),
        helper.make_node('Add', ['mm_out', 'B'], ['Y'], name='add_0'),
    ]
    graph = helper.make_graph(nodes, 'linear', [X], [Y], initializer=[W, B])
    return helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])


def insert_node_after(model, target_output, op_type, name):
    """Insert a unary node after target_output, rewiring all consumers."""
    graph = model.graph
    new_out = f"{name}_out"
    for node in graph.node:
        if target_output in list(node.output):
            continue
        for i, inp in enumerate(node.input):
            if inp == target_output:
                node.input[i] = new_out
    for out in graph.output:
        if out.name == target_output:
            out.name = new_out
    new_node = helper.make_node(op_type, [target_output], [new_out], name=name)
    idx = next((i+1 for i, n in enumerate(graph.node) if target_output in list(n.output)), 0)
    graph.node.insert(idx, new_node)
    return model


model_relu = insert_node_after(build_simple_model(), 'mm_out', 'Relu', 'relu_ins')
validate_model(model_relu, "After ReLU insertion")
for n in model_relu.graph.node:
    print(f"  {n.op_type}: {list(n.input)} -> {list(n.output)}")

## 4. Node Deletion Pattern

To delete node $w$ (input $t$, output $t'$):
1. Replace all references to $t'$ with $t$
2. Update graph outputs if needed
3. Remove $w$ from node list

```
BEFORE:                           AFTER:

 ┌───┐  t  ┌───┐  t' ┌───┐      ┌───┐       t       ┌───┐
 │ u │────►│ w │────►│ v │      │ u │───────────────►│ v │
 └───┘     └───┘     └───┘      └───┘                └───┘
            (del)                (w removed, v uses t)
```

In [ ]:
def build_model_with_identities():
    X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
    Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 3])
    W = numpy_helper.from_array(np.random.randn(4, 3).astype(np.float32), name='W')
    B = numpy_helper.from_array(np.random.randn(3).astype(np.float32), name='B')
    nodes = [
        helper.make_node('Identity', ['X'], ['id1_out'], name='id_1'),
        helper.make_node('MatMul', ['id1_out', 'W'], ['mm_out'], name='mm'),
        helper.make_node('Identity', ['mm_out'], ['id2_out'], name='id_2'),
        helper.make_node('Add', ['id2_out', 'B'], ['add_out'], name='add'),
        helper.make_node('Identity', ['add_out'], ['Y'], name='id_3'),
    ]
    graph = helper.make_graph(nodes, 'with_ids', [X], [Y], initializer=[W, B])
    return helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])


def remove_identity_nodes(model):
    """Remove all Identity nodes by bypassing them."""
    graph = model.graph
    to_remove = []
    for node in graph.node:
        if node.op_type != 'Identity':
            continue
        inp, out = node.input[0], node.output[0]
        for other in graph.node:
            if other == node:
                continue
            for i, x in enumerate(other.input):
                if x == out:
                    other.input[i] = inp
        for g_out in graph.output:
            if g_out.name == out:
                g_out.name = inp
        to_remove.append(node)
    for n in to_remove:
        graph.node.remove(n)
    return model, len(to_remove)


model_id = build_model_with_identities()
print(f"Before: {[n.op_type for n in model_id.graph.node]}")
model_clean, cnt = remove_identity_nodes(model_id)
print(f"Removed {cnt} Identity nodes")
print(f"After: {[n.op_type for n in model_clean.graph.node]}")
validate_model(model_clean, "Identity removal")

In [ ]:
def eliminate_dead_nodes(model):
    """Remove nodes whose outputs are never consumed."""
    graph = model.graph
    consumed = set()
    for node in graph.node:
        for inp in node.input:
            if inp:
                consumed.add(inp)
    for out in graph.output:
        consumed.add(out.name)
    total = 0
    changed = True
    while changed:
        changed = False
        for node in list(graph.node):
            if not any(o in consumed for o in node.output if o):
                graph.node.remove(node)
                for inp in node.input:
                    consumed.discard(inp)
                changed, total = True, total + 1
    return model, total


X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 4])
nodes = [
    helper.make_node('Relu', ['X'], ['relu_out'], name='relu'),
    helper.make_node('Sigmoid', ['X'], ['sig_out'], name='dead_sig'),
    helper.make_node('Tanh', ['sig_out'], ['tanh_out'], name='dead_tanh'),
    helper.make_node('Identity', ['relu_out'], ['Y'], name='out'),
]
graph = helper.make_graph(nodes, 'dead_demo', [X], [Y])
model_d = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])
print(f"Before: {[n.op_type for n in model_d.graph.node]}")
model_d, removed = eliminate_dead_nodes(model_d)
print(f"After (removed {removed}): {[n.op_type for n in model_d.graph.node]}")

## 5. Operator Replacement

Replace an operator with a semantically equivalent subgraph:

$$\text{Sub}(A, B) \equiv \text{Add}(A, \text{Neg}(B))$$
$$\text{Div}(A, B) \equiv \text{Mul}(A, \text{Reciprocal}(B))$$

| Original | Replacement | Rationale |
|----------|-------------|-----------|
| `Sub(A,B)` | `Add(A, Neg(B))` | HW without Sub |
| `Div(A,B)` | `Mul(A, Reciprocal(B))` | Faster on accelerators |
| `Gemm` | `MatMul + Add` | Wider support |

In [ ]:
def build_sub_model():
    X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
    Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 4])
    C = numpy_helper.from_array(np.array([1., 2., 3., 4.], dtype=np.float32), name='C')
    sub = helper.make_node('Sub', ['X', 'C'], ['Y'], name='sub_0')
    graph = helper.make_graph([sub], 'sub_model', [X], [Y], initializer=[C])
    return helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])


def replace_sub_with_add_neg(model):
    """Replace Sub(A, B) with Add(A, negated_B) for constant B."""
    graph = model.graph
    init_names = {i.name for i in graph.initializer}
    to_remove, to_add = [], []
    for node in graph.node:
        if node.op_type != 'Sub':
            continue
        a, b = node.input[0], node.input[1]
        if b in init_names:
            for init in graph.initializer:
                if init.name == b:
                    neg = numpy_helper.from_array(-numpy_helper.to_array(init), f"{b}_neg")
                    graph.initializer.append(neg)
                    break
            add = helper.make_node('Add', [a, f"{b}_neg"], [node.output[0]], name=f"{node.name}_add")
            to_remove.append(node)
            to_add.append((node, add))
    for orig in to_remove:
        idx = list(graph.node).index(orig)
        graph.node.remove(orig)
        for o, new in to_add:
            if o == orig:
                graph.node.insert(idx, new)
    return model


def verify_equivalence(model_a, model_b, inputs, rtol=1e-5, atol=1e-6):
    sa = ort.InferenceSession(model_a.SerializeToString())
    sb = ort.InferenceSession(model_b.SerializeToString())
    oa, ob = sa.run(None, inputs), sb.run(None, inputs)
    for i, (a, b) in enumerate(zip(oa, ob)):
        diff = np.max(np.abs(a - b))
        status = "MATCH" if np.allclose(a, b, rtol=rtol, atol=atol) else "MISMATCH"
        print(f"  Output {i}: {status} (max diff: {diff:.2e})")
    return all(np.allclose(a, b, rtol=rtol, atol=atol) for a, b in zip(oa, ob))


orig_sub = build_sub_model()
mod_add = replace_sub_with_add_neg(copy.deepcopy(orig_sub))
print("Before:", [n.op_type for n in orig_sub.graph.node])
print("After: ", [n.op_type for n in mod_add.graph.node])
validate_model(mod_add, "Sub→Add")
print("Equivalence:")
verify_equivalence(orig_sub, mod_add, {'X': np.random.randn(1, 4).astype(np.float32)})

## 6. Model Merging (`onnx.compose`)

**Sequential composition** $G_1 \circ G_2$: output of $G_1$ feeds input of $G_2$:

$$\text{compose}(G_1, G_2, \text{io\_map}) = G_{\text{merged}}$$

```
┌────────────────────────────────────────────────────┐
│            Model Merging Pipeline                   │
├────────────────────────────────────────────────────┤
│  ┌───────────┐     io_map      ┌───────────┐      │
│  │  Model A  │════════════════►│  Model B  │      │
│  │(prefixed) │ A_out ► B_in    │(prefixed) │      │
│  └───────────┘                  └───────────┘      │
│       │                               │            │
│       ▼                               ▼            │
│  ┌──────────────────────────────────────────┐      │
│  │      Merged Model (validated)            │      │
│  │  • No name collisions (add_prefix)       │      │
│  │  • Shape inference propagated            │      │
│  └──────────────────────────────────────────┘      │
└────────────────────────────────────────────────────┘
```

In [ ]:
def build_encoder():
    X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
    Z = helper.make_tensor_value_info('Z', TensorProto.FLOAT, [1, 2])
    W = numpy_helper.from_array(np.random.randn(4, 2).astype(np.float32), name='W_enc')
    mm = helper.make_node('MatMul', ['X', 'W_enc'], ['Z'], name='encode')
    graph = helper.make_graph([mm], 'encoder', [X], [Z], initializer=[W])
    return helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])

def build_decoder():
    H = helper.make_tensor_value_info('H', TensorProto.FLOAT, [1, 2])
    Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 4])
    W = numpy_helper.from_array(np.random.randn(2, 4).astype(np.float32), name='W_dec')
    mm = helper.make_node('MatMul', ['H', 'W_dec'], ['dec_out'], name='decode')
    sig = helper.make_node('Sigmoid', ['dec_out'], ['Y'], name='activate')
    graph = helper.make_graph([mm, sig], 'decoder', [H], [Y], initializer=[W])
    return helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])

encoder, decoder = build_encoder(), build_decoder()
merged = compose.merge_models(encoder, decoder, io_map=[('Z', 'H')])

print(f"Merged: inputs={[i.name for i in merged.graph.input]}, "
      f"outputs={[o.name for o in merged.graph.output]}")
print(f"Nodes: {[n.op_type for n in merged.graph.node]}")
validate_model(merged, "Merged autoencoder")

sess = ort.InferenceSession(merged.SerializeToString())
x = np.random.randn(1, 4).astype(np.float32)
print(f"Input: {x[0]}, Output: {sess.run(None, {'X': x})[0][0]}")

In [ ]:
# add_prefix for name collision avoidance
def build_relu_block(s):
    X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
    Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 4])
    graph = helper.make_graph(
        [helper.make_node('Relu', ['X'], ['Y'], name=f'relu_{s}')],
        f'block_{s}', [X], [Y])
    return helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])

a = compose.add_prefix(build_relu_block('a'), prefix='A/')
b = compose.add_prefix(build_relu_block('b'), prefix='B/')
print(f"A: {[i.name for i in a.graph.input]} -> {[o.name for o in a.graph.output]}")
print(f"B: {[i.name for i in b.graph.input]} -> {[o.name for o in b.graph.output]}")

seq = compose.merge_models(a, b, io_map=[('A/Y', 'B/X')])
print(f"Sequential merge: {[n.op_type for n in seq.graph.node]} nodes")
validate_model(seq, "Sequential")

## 7. Safe Rewrite Patterns

A rewrite $G \to G'$ is safe iff:

$$\text{valid}(G') \wedge \text{shape\_consistent}(G') \wedge \left(\forall x: \|f_G(x) - f_{G'}(x)\| < \epsilon\right)$$

Safety protocol:
1. `onnx.checker.check_model()` — structural validity
2. `shape_inference.infer_shapes()` — type/shape propagation
3. Numerical equivalence on random inputs

In [ ]:
def safe_rewrite(original, rewrite_fn, test_inputs, name="rewrite"):
    """Apply rewrite with full safety validation."""
    model = rewrite_fn(copy.deepcopy(original))
    try:
        checker.check_model(model)
        print(f"[{name}] Step 1 PASS: Structure")
    except Exception as e:
        print(f"[{name}] Step 1 FAIL: {e}")
        return None
    try:
        model = shape_inference.infer_shapes(model)
        print(f"[{name}] Step 2 PASS: Shapes")
    except Exception as e:
        print(f"[{name}] Step 2 FAIL: {e}")
        return None
    so = ort.InferenceSession(original.SerializeToString())
    sm = ort.InferenceSession(model.SerializeToString())
    oo, om = so.run(None, test_inputs), sm.run(None, test_inputs)
    if all(np.allclose(a, b, rtol=1e-5, atol=1e-6) for a, b in zip(oo, om)):
        print(f"[{name}] Step 3 PASS: Equivalence")
        print(f"[{name}] SAFE ✓")
        return model
    print(f"[{name}] Step 3 FAIL: Output mismatch")
    return None


model_ids = build_model_with_identities()
data = {'X': np.random.randn(1, 4).astype(np.float32)}
safe_rewrite(model_ids, lambda m: remove_identity_nodes(m)[0], data, "Id-removal")

## 8. Rewrite Pass: Bias Folding (MatMul+Add → Gemm)

$$\text{Add}(\text{MatMul}(X, W), B) \to \text{Gemm}(X, W, B)$$

Preconditions: $B$ is 1-D constant, MatMul output has single consumer, $\dim(B) = \text{cols}(W)$.

In [ ]:
def fold_matmul_add_to_gemm(model):
    """Fold MatMul + Add(bias) into Gemm."""
    graph = model.graph
    init_names = {i.name for i in graph.initializer}
    out_to_node = {o: n for n in graph.node for o in n.output}
    consumers = {}
    for n in graph.node:
        for inp in n.input:
            if inp:
                consumers[inp] = consumers.get(inp, 0) + 1

    to_remove, to_add = [], []
    for node in graph.node:
        if node.op_type != 'Add':
            continue
        mm_in = bias_in = None
        for inp in node.input:
            if inp in out_to_node and out_to_node[inp].op_type == 'MatMul':
                mm_in = inp
            elif inp in init_names:
                bias_in = inp
        if not mm_in or not bias_in or consumers.get(mm_in, 0) != 1:
            continue
        mm = out_to_node[mm_in]
        gemm = helper.make_node('Gemm', [mm.input[0], mm.input[1], bias_in],
                                [node.output[0]], name=f"{mm.name}_gemm",
                                alpha=1.0, beta=1.0, transB=0)
        to_remove.extend([mm, node])
        to_add.append(gemm)

    for n in to_remove:
        if n in graph.node:
            graph.node.remove(n)
    for n in to_add:
        graph.node.append(n)
    print(f"Folded {len(to_add)} MatMul+Add -> Gemm")
    return model


orig = build_simple_model()
data = {'X': np.random.randn(1, 4).astype(np.float32)}
print("Before:", [n.op_type for n in orig.graph.node])
folded = safe_rewrite(orig, fold_matmul_add_to_gemm, data, "Bias-fold")
if folded:
    print("After:", [n.op_type for n in folded.graph.node])

## 9. Multi-Rule Rewrite Engine

A pattern-matching engine applies multiple rules to fixed point:

```
┌──────────────────────────────────────────┐
│       Multi-Pattern Rewrite Engine        │
├──────────────────────────────────────────┤
│  ┌────────┐ ┌────────┐ ┌────────┐       │
│  │Sub→Add │ │Div→Mul │ │ Custom │       │
│  └───┬────┘ └───┬────┘ └───┬────┘       │
│      └──────────┼──────────┘             │
│                 ▼                        │
│        [Match & Apply] ◄─ fixpoint       │
│                 ▼                        │
│           [Validate]                     │
└──────────────────────────────────────────┘
```

In [ ]:
class RewriteRule:
    def match(self, node, ctx): raise NotImplementedError
    def apply(self, node, ctx): raise NotImplementedError

class SubToAddNeg(RewriteRule):
    def match(self, node, ctx):
        return node.op_type == 'Sub' and node.input[1] not in ctx['inits']
    def apply(self, node, ctx):
        a, b, out = node.input[0], node.input[1], node.output[0]
        neg = helper.make_node('Neg', [b], [f"{b}_neg"], name=f"{node.name}_neg")
        add = helper.make_node('Add', [a, f"{b}_neg"], [out], name=f"{node.name}_add")
        return [node], [neg, add]

class DivToMulRecip(RewriteRule):
    def match(self, node, ctx):
        return node.op_type == 'Div'
    def apply(self, node, ctx):
        a, b, out = node.input[0], node.input[1], node.output[0]
        recip = helper.make_node('Reciprocal', [b], [f"{b}_rcp"], name=f"{node.name}_rcp")
        mul = helper.make_node('Mul', [a, f"{b}_rcp"], [out], name=f"{node.name}_mul")
        return [node], [recip, mul]


def apply_rules(model, rules, max_iter=10):
    graph = model.graph
    total = 0
    for _ in range(max_iter):
        ctx = {'inits': {i.name for i in graph.initializer}}
        applied = 0
        for node in list(graph.node):
            if node not in graph.node:
                continue
            for rule in rules:
                if rule.match(node, ctx):
                    rm, add = rule.apply(node, ctx)
                    idx = list(graph.node).index(rm[0])
                    for n in rm:
                        graph.node.remove(n)
                    for i, n in enumerate(add):
                        graph.node.insert(idx + i, n)
                    applied += 1
                    break
        total += applied
        if not applied:
            break
    print(f"Applied {total} rewrite(s)")
    return model

In [ ]:
X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
B = helper.make_tensor_value_info('B', TensorProto.FLOAT, [1, 4])
Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 4])
nodes = [
    helper.make_node('Sub', ['X', 'B'], ['sub_out'], name='sub'),
    helper.make_node('Div', ['sub_out', 'B'], ['Y'], name='div'),
]
graph = helper.make_graph(nodes, 'sd', [X, B], [Y])
model_sd = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])

print("Before:", [n.op_type for n in model_sd.graph.node])
rewritten = apply_rules(copy.deepcopy(model_sd), [SubToAddNeg(), DivToMulRecip()])
print("After:", [n.op_type for n in rewritten.graph.node])
validate_model(rewritten, "Rewritten")

tx = np.random.randn(1, 4).astype(np.float32)
tb = np.abs(np.random.randn(1, 4).astype(np.float32)) + 0.1
print("Equivalence:")
verify_equivalence(model_sd, rewritten, {'X': tx, 'B': tb})

## 10. Complete Optimization Pipeline

```
┌────────┐   ┌──────────┐   ┌─────────┐   ┌────────┐   ┌──────────┐
│ Input  │──►│ Identity │──►│Operator │──►│  Bias  │──►│ Validate │
│ Model  │   │ Removal  │   │ Replace │   │Folding │   │ & Export │
└────────┘   └──────────┘   └─────────┘   └────────┘   └──────────┘
```

In [ ]:
def build_complex_model():
    X = helper.make_tensor_value_info('X', TensorProto.FLOAT, [1, 4])
    Y = helper.make_tensor_value_info('Y', TensorProto.FLOAT, [1, 3])
    W = numpy_helper.from_array(np.random.randn(4, 3).astype(np.float32), name='W')
    B = numpy_helper.from_array(np.random.randn(3).astype(np.float32), name='B')
    ones = numpy_helper.from_array(np.ones(3, dtype=np.float32), name='ones')
    nodes = [
        helper.make_node('Identity', ['X'], ['x_id'], name='id_in'),
        helper.make_node('MatMul', ['x_id', 'W'], ['mm'], name='matmul'),
        helper.make_node('Identity', ['mm'], ['mm_id'], name='id_mid'),
        helper.make_node('Add', ['mm_id', 'B'], ['add_out'], name='bias_add'),
        helper.make_node('Sub', ['add_out', 'ones'], ['sub_out'], name='sub_ones'),
        helper.make_node('Identity', ['sub_out'], ['Y'], name='id_out'),
    ]
    graph = helper.make_graph(nodes, 'complex', [X], [Y], initializer=[W, B, ones])
    return helper.make_model(graph, opset_imports=[helper.make_opsetid('', 17)])


def run_pipeline(model):
    print(f"Initial ({len(model.graph.node)} nodes): {[n.op_type for n in model.graph.node]}")
    model, n = remove_identity_nodes(model)
    print(f"After id-removal (-{n}): {[n.op_type for n in model.graph.node]}")
    model = replace_sub_with_add_neg(model)
    print(f"After Sub→Add: {[n.op_type for n in model.graph.node]}")
    model = fold_matmul_add_to_gemm(model)
    print(f"Final ({len(model.graph.node)} nodes): {[n.op_type for n in model.graph.node]}")
    validate_model(model, "Optimized")
    return model


original = build_complex_model()
data = {'X': np.random.randn(1, 4).astype(np.float32)}
optimized = run_pipeline(copy.deepcopy(original))
print("\nEnd-to-end equivalence:")
verify_equivalence(original, optimized, data)

## Summary

| Technique | Key Insight |
|-----------|------------|
| **Rewrite Rules** | $G[p/r] = G'$ with $\forall x: f_G(x) = f_{G'}(x)$ |
| **Invariants** | 4 predicates must hold after every modification |
| **Node Insertion** | Create new value, rewire ALL consumers |
| **Node Deletion** | Bypass predecessor output to all consumers |
| **Op Replacement** | Maintain I/O contract, migrate attributes |
| **Model Merging** | `compose.merge_models` + `io_map` + `add_prefix` |
| **Safe Rewrites** | Validate → Shape Infer → Test Equivalence |

**Rules of thumb**: always `deepcopy` before modifying, validate after every pass,
test numerically, and watch for multi-consumer edges.